# Customer Dashboard

In [1]:
!pip install dash


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import dash
from dash import dcc, html, Input, Output
import webbrowser

# =========================================================
# LOAD DATA
# =========================================================
base = Path("../data/gold")

df_orders = pd.read_csv(base / "customer_orders.csv")
df_country = pd.read_csv(base / "customer_country.csv")
df_activity = pd.read_csv(base / "customer_activity.csv")
df_value = pd.read_csv(base / "customer_value.csv")
df_rfm = pd.read_csv(base / "rfm_scores.csv")

# =========================================================
# CLEAN COLUMNS
# =========================================================
for df in [df_orders, df_country, df_activity, df_value, df_rfm]:
    df.columns = df.columns.str.strip()

# =========================================================
# DATE PREP
# =========================================================
df_orders["OrderDate"] = pd.to_datetime(df_orders["OrderDate"], errors="coerce")
df_orders["Year"] = df_orders["OrderDate"].dt.year
df_orders["Month"] = df_orders["OrderDate"].dt.month_name()

# =========================================================
# MERGE DATA
# =========================================================
df_main = df_value.merge(
    df_country[["CustomerID", "Country"]],
    on="CustomerID",
    how="left"
)

# =========================================================
# FILTER OPTIONS
# =========================================================
segment_options = [
    {"label": s, "value": s}
    for s in sorted(df_main["Segment"].dropna().unique())
]

country_options = [
    {"label": c, "value": c}
    for c in sorted(df_main["Country"].dropna().unique())
]

year_options = [
    {"label": y, "value": y}
    for y in sorted(df_orders["Year"].dropna().unique())
]

# =========================================================
# APP
# =========================================================
app = dash.Dash(__name__)
server = app.server

# =========================================================
# STYLES
# =========================================================
page_style = {
    "backgroundColor": "#0B1020",
    "minHeight": "100vh",
    "fontFamily": "Segoe UI"
}

card_style = {
    "backgroundColor": "#131C31",
    "borderRadius": "18px",
    "padding": "15px"
}

chart_style = {
    "backgroundColor": "#131C31",
    "borderRadius": "18px",
    "padding": "15px"
}

label_style = {
    "color": "white",
    "fontWeight": "600"
}

kpi_style = {
    "padding": "18px",
    "borderRadius": "18px",
    "color": "white",
    "textAlign": "center",
    "background": "linear-gradient(135deg,#4facfe,#00f2fe)"
}

# =========================================================
# LAYOUT
# =========================================================
app.layout = html.Div(style=page_style, children=[

    # TITLE
    html.H1(
        "👥 Customer Segmentation Dashboard",
        style={"textAlign": "center", "color": "white", "padding": "40px"}
    ),

    # FILTERS
    html.Div(style={"width": "92%", "margin": "auto"}, children=[

        html.Div(style=card_style, children=[

            html.Div(style={"display": "grid", "gridTemplateColumns": "1fr 1fr 1fr", "gap": "15px"}, children=[

                html.Div([
                    html.Label("Segment", style=label_style),
                    dcc.Dropdown(
                        id="segment",
                        options=segment_options,
                        multi=True
                    )
                ]),

                html.Div([
                    html.Label("Country", style=label_style),
                    dcc.Dropdown(
                        id="country",
                        options=country_options,
                        multi=True
                    )
                ]),

                html.Div([
                    html.Label("Year", style=label_style),
                    dcc.Dropdown(
                        id="year",
                        options=year_options,
                        value=2011,
                        clearable=False
                    )
                ])
            ])
        ])
    ]),

    # =====================================================
    # KPI (3 TILES)
    # =====================================================
    html.Div(style={
        "display": "grid",
        "gridTemplateColumns": "repeat(3, 1fr)",
        "gap": "18px",
        "width": "92%",
        "margin": "20px auto"
    }, children=[

        html.Div(id="kpi_revenue", style=kpi_style),

        html.Div(id="kpi_customers", style=kpi_style),

        html.Div(id="kpi_orders", style=kpi_style),

    ]),

    # =====================================================
    # CHART GRID
    # =====================================================
    html.Div(style={
        "display": "grid",
        "gridTemplateColumns": "0.8fr 0.8fr",
        "gap": "20px",
        "width": "92%",
        "margin": "auto"
    }, children=[

        html.Div(style=chart_style, children=[
            html.H3("🧩 Customer Segment Breakdown", style={"color": "white"}),
            dcc.Graph(id="segment_breakdown")
        ]),

        html.Div(style=chart_style, children=[
            html.H3("💰 AOV Distribution", style={"color": "white"}),
            dcc.Graph(id="aov_chart")
        ]),

        html.Div(style=chart_style, children=[
            html.H3("📈 Revenue Trend", style={"color": "white"}),
            dcc.Graph(id="trend_chart")
        ]),

        html.Div(style=chart_style, children=[
            html.H3("🔥 RFM Heatmap", style={"color": "white"}),
            dcc.Graph(id="heatmap")
        ])
    ])
])

# =========================================================
# CALLBACK
# =========================================================
@app.callback(
    Output("segment_breakdown", "figure"),
    Output("aov_chart", "figure"),
    Output("trend_chart", "figure"),
    Output("heatmap", "figure"),

    Output("kpi_revenue", "children"),
    Output("kpi_customers", "children"),
    Output("kpi_orders", "children"),

    Input("segment", "value"),
    Input("country", "value"),
    Input("year", "value")
)
def update_dashboard(segments, countries, year):

    df_filtered = df_main.copy()

    if segments:
        df_filtered = df_filtered[df_filtered["Segment"].isin(segments)]

    if countries:
        df_filtered = df_filtered[df_filtered["Country"].isin(countries)]

    orders_filtered = df_orders[df_orders["Year"] == year]

    # =====================================================
    # KPI VALUES
    # =====================================================
    revenue = df_filtered["TotalRevenue"].sum()
    customers = df_filtered["CustomerID"].nunique()
    orders = df_filtered["OrderCount"].sum()

    kpi_revenue = [
        html.H4("💰 Revenue"),
        html.H2(f"${revenue:,.0f}")
    ]

    kpi_customers = [
        html.H4("👥 Customers"),
        html.H2(f"{customers:,}")
    ]

    kpi_orders = [
        html.H4("🛒 Orders"),
        html.H2(f"{orders:,}")
    ]

    # =====================================================
    # SEGMENT BREAKDOWN (PIE)
    # =====================================================
    seg = df_filtered["Segment"].value_counts().reset_index()
    seg.columns = ["Segment", "Count"]

    fig1 = px.pie(
        seg,
        names="Segment",
        values="Count",
        color="Segment",
        color_discrete_map={
            "High Value": "#6C5CE7",
            "Mid Value": "#00CEC9",
            "Low Value": "#E17055"
        }
    )

    fig1.update_traces(textinfo="label+percent")

    # =====================================================
    # AOV
    # =====================================================
    fig2 = px.histogram(df_filtered, x="AOV", color="Segment", nbins=30)

    # =====================================================
    # TREND
    # =====================================================
    trend = orders_filtered.groupby(
        orders_filtered["OrderDate"].dt.to_period("M")
    )["OrderRevenue"].sum().reset_index()

    trend["OrderDate"] = trend["OrderDate"].astype(str)

    fig3 = px.line(trend, x="OrderDate", y="OrderRevenue", markers=True)

    # =====================================================
    # HEATMAP
    # =====================================================
    corr = df_rfm[
        ["Recency", "Frequency", "Monetary", "R_rank", "F_rank", "M_rank", "RFM_Score"]
    ].corr()

    fig4 = px.imshow(corr, text_auto=True, color_continuous_scale="Purples")

    return fig1, fig2, fig3, fig4, kpi_revenue, kpi_customers, kpi_orders

# =========================================================
# RUN APP
# =========================================================
# =========================================================
if __name__ == "__main__":

    webbrowser.open("http://127.0.0.1:8050")

    app.run(
        host="127.0.0.1",
        port=8050,
        debug=False
    )